# 24 · 和通用大模型比：最强的模型不给工具，够不够？

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → 17 找相似病人 ✅ → 18 疾病介绍（RAG） ✅ → 19 基因工具 ✅ → 20 大考 ✅ → 21 正式评测 ✅ → 22 做成服务 ✅ → 23 和别人比 ✅ → [24 和大模型比 ✅]
```

> **这一章的代码在合并请求 #26 里。** 三家模型通过 OpenRouter 调用，一共花了 **$4.90**（其中 $2.59 花在两次作废的考试上，第 5 步讲原因）。运行这本笔记不花钱，只读记录。

## 背景：如果最强的模型直接问就行，agent 还有什么用？

我们的 agent 用的是一个便宜的模型（DeepSeek `deepseek-flash`，第 21 章算过一道题约 ¥0.02），靠的是工具和流程。

任何一个产品经理都会先问：**直接用最强的模型，什么都不给，问一句，行不行？** 如果行，前面 23 章做的工具、agent、评测，大半就没有必要了。

这一章就回答这个问题：OpenAI、Anthropic、Google 各取一个当前主力模型，在留出集上直接问，和 agent 逐题比。

**公司里的做法**：这叫"先和最简单的方案比"。要上一个复杂的系统，必须先证明它比"直接调最好的模型"强，而且强得值得那份复杂。

## 这一章做的 7 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 一个地方调三家 | OpenRouter：兼容 OpenAI 的统一接口 | 不用开三个账户，代码不用改 |
| 2 | 花钱之前先设上限 | 每次调用都记账，到上限就停 | 一次考试不会花超 |
| 3 | 小考 | 每家 20 道开发题 | 量出真实单价；提前踩坑 |
| 4 | 先写方案 | 预注册；固定的 200 道题 | 只花对手的钱，比较照样成立 |
| 5 | 正式考试，和补问 | `--fill-in`：只补问因接口故障没答出来的题 | 不为网络问题重花一遍钱 |
| 6 | 结果 | 逐题配对检验 | 回答方案里的三个主要问题 |
| 7 | 怎么理解这个结果 | | agent 的价值到底在哪 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
import logging, warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # 一条和我们无关的提示
logging.getLogger("mlflow").setLevel(logging.WARNING)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 一个地方调三家

### 在干什么
用一个账户、一份余额，调用三家公司的模型。

### 背景
三家各有各的账户、计费、接口格式。各开一个，要充三次钱、管三把密钥、写三套调用代码。

### 用了什么技术，为什么
**OpenRouter**：一个"中转站"。它对外提供和 OpenAI 一样的接口，背后转给几百个模型。我们的代码从第 09 章起就用 OpenAI 的标准客户端，换一个地址和密钥就能调任何一家。打个比方：一张能在各家超市用的购物卡。

代码里只多了三样东西（`src/raredx/llm.py`、`src/raredx/baselines/llm_direct.py`）：OpenRouter 的地址和密钥（密钥只放 `.env`）；"直接猜"这个对照可以指定模型（`--model openrouter:openai/gpt-6.1-sol`）；每次调用要求对方报出这一次花了多少钱。

**考法和第 09、21 章的"大模型直接猜"完全一样**：同一份提示词（`direct-v1`），一道题问一次，要它列 10 个最可能的病和 OMIM 编号。

### 公司里什么情况怎么做
- **模型网关**（LLM gateway）：公司里通常有一个统一的入口，所有团队通过它调各家模型，统一记账、限流、留日志。OpenRouter 就是一个外部的网关
- **换模型只改配置**：业务代码不应该和某一家的接口绑死

### 面试怎么说
> "Every model call goes through the OpenAI SDK against an OpenAI-compatible endpoint, so comparing OpenAI, Anthropic and Google models was configuration — OpenRouter as the gateway — plus a cost report on every call."

---

## 第 2 步 · 花钱之前先设上限

### 在干什么
保证一次考试不会花得比预计多。

### 背景
第 20 章有一次教训：一次判分实验事先估计约 ¥3、上限 ¥10，实际开了"深度思考"，一次就花了约 ¥19。当时的上限是"考完一次看一次"，挡不住一次考试本身花超。

### 用了什么技术，为什么
**把上限放进每一次调用里**（`llm.Budget`）：

1. 每次调用之前，先看已经花了多少；到了上限，就不再调用
2. 每次调用之后，把 OpenRouter 报出的这一次的花费加上去
3. 被上限挡下的题，算"接口故障"，不算模型答错：这次考试作废，而不是给出一个被低估的分数

还有一条：**花钱的考试，不写上限就不许开始**。命令行在读数据之前就检查，没有 `--budget-usd` 直接拒绝。

### 公司里什么情况怎么做
- 各家模型平台都可以给密钥设额度；公司内部的网关会按团队、按项目设预算和告警
- 批量任务一定要有"熔断"：花到上限就停，不能等月底看账单

### 执行

In [ ]:
run("uv run raredx eval run --system llm-direct --model openrouter:openai/gpt-6.1-sol --split ci-20", check=False)

### 结果怎么看
命令直接被拒绝：`a run billed through OpenRouter needs a cap: give --budget-usd`。还没读任何数据，更没花钱。

### 面试怎么说
> "A cap checked between runs can't stop one run that costs more than expected — I learned that when a 'three-yuan' experiment cost nineteen. So the budget is checked before every model call, and a billed run without a cap is refused before it reads any data."

---

## 第 3 步 · 小考：每家 20 道开发题

### 在干什么
正式考试之前，每家先在 20 道开发题（ci-20）上考一次，每次上限 $0.5。

### 背景
单价不能靠估：每家模型回答的长短不一样，有的还会偷偷"思考"（思考的字数也收费）。只有真跑一次才知道。

### 结果怎么看：三个真实踩过的坑

> **坑 1：关不掉的"深度思考"**
>
> 方案里写的是"关掉深度思考"。第一次小考，GPT 的 20 道题全部失败，接口返回：`Reasoning is mandatory for this endpoint and cannot be disabled`（这个接口必须思考，不能关）。三家都是这样。好在请求被拒绝时不扣钱。
>
> 改成三家统一用"最低档思考"（`effort = minimal`）。在一道简单的题上，最低档三家都不产生思考的字数。方案里加了一条注明日期的说明：这是在碰留出集之前改的。

> **坑 2：新账户每分钟只能调 20 次**
>
> Claude 的小考，20 道题却记下了 46 条记录，被我们的"记录完整"检查判为无效。原因：OpenRouter 对新账户调 Claude 限制每分钟 20 次，超了就返回 429；MLflow 的考试程序遇到 429 会自动重试，**每重试一次就多记一条**。
>
> 改法：每道题以最后一次尝试为准（`Harness.answers`），并补了一个测试（模拟一个每道题第一次都被限流的服务）。

> **坑 3：账户余额的数字会晚几分钟**
>
> 本来用"考前余额减考后余额"算一次考试花了多少。三次小考一共花了 $0.41，刚考完时余额只显示花了 $0.29：余额要过几分钟才更新。改成直接把每次调用报出的花费加起来，这个数是即时、准确的。

**小考量出来的单价**（200 道的估计）：GPT 约 $1.4，Claude 约 $1.6，Gemini 约 $1.1。

### 面试怎么说
> "A twenty-case pilot per model before any billed run on held-out data found three things the docs didn't tell me: the current models refuse to turn reasoning off, a new account is rate-limited to twenty requests a minute — and MLflow's evaluator records every retried attempt — and the account balance lags the per-call cost by minutes."

---

## 第 4 步 · 先写方案

### 在干什么
和前两次一样，在碰留出集之前写好方案：[`docs/comparison_plan_llms.md`](../comparison_plan_llms.md)。

### 用了什么技术，为什么

**题目：固定的 200 道**（`evals/splits/heldout-papers-200.tsv`），从第 21 章的 840 道里用固定种子随机抽。

为什么不考 840 道：钱。三家各考 840 道要 $15 以上，超出预算。为什么 200 道就够：200 道题，能分出 10 个百分点左右的差距；更重要的是，**agent 和其他系统在这 200 道上的答案早就记录了**，不用重跑。只花对手的钱，逐题配对照样能做。

这 200 道是不是有代表性？agent 在这 200 道上第 1 名 43.0%，在全部 840 道上 42.4%，几乎一样。

**每个模型一个主要问题**：agent 的第 1 名是不是显著高于它。

**一个管不住的不公平，事先写明**：这些病例都发表过，大模型训练时可能读过，答对可能是"记得"。我们没法替模型遮挡它记得的东西，所以模型的分数是偏高的上限。这对 agent 不利，不会有利。

### 面试怎么说
> "Comparisons billed per case run on a fixed random 200 of the 840 held-out cases — representative: the agent scores 43.0% there against 42.4% on all 840 — and since every system already run on the 840 has its answers to those 200 recorded, only the new model is paid for."

---

## 第 5 步 · 正式考试，和补问

### 在干什么
三家各考 200 道。两家第一次没考完整，只补问没答出来的题。

### 背景
- **GPT-6.1**：一次完成，$1.44
- **Gemini 4**：35 道撞上"每分钟 20 次"的限制，MLflow 重试三次仍不行，没答出来。它答得太快（每道 1.7 秒），4 路同时问，每分钟远超 20 次
- **Claude Sonnet 5.5**：已经改成 1 路慢慢问，28 道却遇到"连接中断"

按方案的规矩，这两次都作废：有题因为接口故障没答出来，分数就不是模型的真实水平。可是整场重考要再花约 $3，而且那 337 道已经答出来的题，重考一遍答案也不会更好或更差。

### 用了什么技术，为什么
**只补问没答出来的题**（`raredx eval run --fill-in <那次考试的编号>`）：

1. 从那次考试的记录里，找出"最后一次尝试因为接口故障失败"的题（`Harness.lost_cases`）
2. 只把这些题再问一次，记成一次新的考试，注明它补的是哪一次
3. 算分时，每道题用最后一次答出来的答案

为什么这样不影响结论：这些题没答出来，原因是限流和断网，和题难不难、模型会不会都无关；再问一次，得到的就是整场重考时它们会得到的答案。补问时把速度压到每分钟 18 次以内，连接中断自动重试 5 次。

**这个改动是在看到部分结果之后做的**，方案里如实写了这一点和理由。改的是"接口出故障怎么办"，不是"看什么、怎么算赢"。

> **一个真实踩过的坑：网络被拦了**
>
> 补问刚开始，这台电脑突然连不上 OpenRouter 了：加密连接的第一步就被重置（`Connection reset by peer`），别的网站都正常。Claude 那 28 道"连接中断"其实就是它的开头。我们这边修不了，只能等网络恢复（后来恢复了）。补问那次 35 道全部失败，好在请求没发出去，不扣钱。

### 结果怎么看
补问各一轮就补齐了：Gemini 35 道全部答出（$0.20），Claude 28 道全部答出（$0.27）。

### 面试怎么说
> "Two runs lost cases to the provider — a per-minute limit on a new account, then dropped connections. Rather than pay to re-ask 337 answered cases, I added a fill-in: re-ask only the cases whose last attempt failed at the API, as a run that completes the original. Those cases were lost for reasons unrelated to their answers, so it measures the same thing; I recorded in the plan that the rule changed after partial results."

---

## 第 6 步 · 结果

### 执行
每道题每个系统答对没有，在 `docs/comparison_llm_hits.tsv`（200 行）。读它：

In [ ]:
import csv
from raredx.evals.scoring import p_value, sign_test, wilson

ROWS = list(csv.DictReader(open(ROOT / "docs/comparison_llm_hits.tsv", encoding="utf-8"), delimiter="\t"))
NAMES = {"llm-direct": "DeepSeek 直接猜", "claude": "Claude Sonnet 5.5", "gemini": "Gemini 4",
         "lirical": "LIRICAL", "gpt": "GPT-6.1", "agent": "最终 agent"}

def rate(s, k, rows=ROWS):
    n = sum(r[f"{s}@{k}"] == "1" for r in rows)
    lo, hi = wilson(n, len(rows))
    return f"{n / len(rows):6.1%} [{lo:.1%}–{hi:.1%}]"

def paired(a, b, k, rows=ROWS):
    w = sum(r[f"{a}@{k}"] == "1" and r[f"{b}@{k}"] == "0" for r in rows)
    l = sum(r[f"{b}@{k}"] == "1" and r[f"{a}@{k}"] == "0" for r in rows)
    return f"只有「{NAMES[a]}」对 {w} 道，只有「{NAMES[b]}」对 {l} 道，{p_value(sign_test(w, l))}"

print(f"留出集里固定的 {len(ROWS)} 道题：")
print(f"{'':<18}{'第 1 名':<24}{'前 10 名'}")
for s in ("llm-direct", "claude", "gemini", "lirical", "gpt", "agent"):
    print(f"{NAMES[s]:<16}{rate(s, 1):<24}{rate(s, 10)}")
print("\n主要结论（第 1 名）：")
for m in ("gpt", "claude", "gemini"):
    print("  ", paired("agent", m, 1))
print("\n前 10 名：")
for m in ("gpt", "claude", "gemini"):
    print("  ", paired("agent", m, 10))
print("\n三家对 LIRICAL（第 1 名）：")
for m in ("gpt", "claude", "gemini"):
    print("  ", paired(m, "lirical", 1))

花费和用时，从 MLflow 的记录里读：

In [ ]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
runs = mlflow.search_runs(experiment_names=["eval/heldout-papers-200"], filter_string="tags.mlflow.runName LIKE 'llm-direct%'")
for _, r in runs.sort_values("start_time").iterrows():
    fills = r.get("params.fills_in")
    fills = fills[:8] if isinstance(fills, str) else "—"
    print(f"{r['tags.mlflow.runName']:<42} 补问的是: {fills:<10}"
          f"花费 ${r['metrics.usage/cost_usd_billed']:.2f}    每道题 {r['metrics.usage/seconds_per_case']:.1f} 秒")

### 结果怎么看

| 200 道留出集的题 | 第 1 名 | 前 10 名 |
|---|---|---|
| DeepSeek 直接猜 | 15.0% | 25.5% |
| Claude Sonnet 5.5 直接猜 | 23.5% | 38.5% |
| Gemini 4 直接猜 | 27.0% | 42.0% |
| LIRICAL | 28.0% | 47.0% |
| GPT-6.1 直接猜 | 33.0% | 49.0% |
| **最终 agent** | **43.0%** | **60.5%** |

**三个主要结论都成立**：agent 对 GPT 是 37 比 17（p = 0.009），对 Claude 52 比 13，对 Gemini 46 比 14（都 p < 0.001）。前 10 名差距更大。

**三家对 LIRICAL 都没有显著差别**：最强的通用模型直接猜，大约就是一个现成的专业工具的水平。

花费那一格：GPT 一次 $1.44。Claude 两行：作废的那次和补问，合计 $1.95。Gemini 三行：作废的那次、断网时一道也没问出去的那次（$0）、补问成功的那次，合计 $1.10。

### 面试怎么说
> "On the same 200 held-out cases, the agent — a cheap model with tools — beats each provider's frontier model asked directly: 43% top-1 against 33% for GPT-6.1, 27% for Gemini 4 and 24% for Claude Sonnet 5.5, all significant paired. The frontier models alone land about where LIRICAL, a specialist tool, does."

---

## 第 7 步 · 怎么理解这个结果

### 在干什么
说清楚这组数字意味着什么，以及不意味着什么。

### 结果怎么看

**1. agent 的优势不是来自模型，是来自工具和流程。** agent 用的模型，自己直接猜只有 15.0%。同一个模型，给它工具、让它按流程查，到了 43.0%，比三家最强的模型直接猜都高。

**2. 三家合起来也不如 agent 一个。** 三家里任何一家第 1 名答对的题，合起来 72 道；agent 一个答对 86 道。

**3. 按一道题的花费算**：agent 约 ¥0.02；GPT-6.1 直接猜约 $0.007（约 ¥0.05），更贵，还更差。

**4. 模型"记得"的那部分**：28 道注定答不对的题（这个病只在病人自己的论文里描述过），三家各答对了 2 道。只有读过那篇论文，或者恰好猜中，才可能答对。这说明直接猜的分数里有一小部分是记忆，所以它们的分数是偏高的上限。

**5. 和考前的预期对照**：

| 预期 | 实际 | 对吗 |
|---|---|---|
| 三家第 1 名都在 20%–35%，比 DeepSeek 直接猜高不少，但都显著低于 agent | 23.5%–33.0% | ✅ |
| 前 10 名差距更大 | 三家都是 | ✅ |
| 三家之间差别不会太大 | GPT 比 Claude 高 9.5 个百分点，逐题 21 比 2 | ❌ |
| 每个模型 200 道花 $1–2 | $1.10–$1.95 | ✅ |

### 公司里什么情况怎么做
- **"换个更强的模型"往往不是答案**：在专业领域，结构化的知识和工具常常比模型大小更重要。这组数字就是一个具体的例子
- **算总账**：比的不只是准不准，还有每次多少钱、多快、能不能解释

### 面试怎么说
> "The model inside the agent scores 15% on its own; with the tools and the workflow it reaches 43%, ahead of every frontier model asked directly. The value is in the tools and the process, not the model — and it's also the cheaper option per case."

---

## 本章小结

| 步骤 | 做了什么 | 结果 |
|---|---|---|
| 一个地方调三家 | OpenRouter，OpenAI 兼容接口 | 只改配置 |
| 花钱的上限 | 每次调用前检查；不写上限不许开始 | 没有一次花超 |
| 小考 | 每家 20 道开发题 | 三个坑：关不掉的思考、每分钟 20 次、余额延迟 |
| 先写方案 | 固定的 200 道；只花对手的钱 | 主要问题：agent 是否胜过每一家 |
| 正式考试 | 两次作废，补问 63 道 | 全部答出 |
| **结果** | 第 1 名 | **agent 43.0%；GPT 33.0%，Gemini 27.0%，Claude 23.5%；三个都显著** |
| 花费 | | OpenRouter $4.90（其中 $2.59 是作废的考试） |

## 一句话

**便宜的模型加上对的工具和流程，胜过最贵的模型直接问。**

## 面试题

1. **Wouldn't a frontier model be enough?**
   > "I tested that: GPT-6.1, Claude Sonnet 5.5 and Gemini 4, asked directly on 200 held-out cases, score 33%, 24% and 27% at first place; the agent, on a far cheaper model, 43% — significant against each, paired."

2. **How did you keep the cost of testing other providers' models under control?**
   > "A per-call budget that stops a run at its cap, a pilot per model to measure unit cost first, a fixed 200-case sample where every other system's answers were already recorded, and a fill-in that re-asks only cases lost to the API instead of re-running everything."

3. **What went wrong?**
   > "Two runs lost cases to the provider — a rate limit on a new account and dropped connections — and were void, about $2.6 of $4.9. I fixed the harness to count retried attempts correctly and added the fill-in, and recorded in the plan that the rule changed after partial results."

**下一章 25 · 和 DeepRare 比**：同一个模型，两种 agent 设计。